# SVM in JAX

This notebook demonstrates a supervised learning workflow using a linear support vector machine.

The example shows margin-based classification trained with hinge loss.

## Theory

A linear SVM learns a separating hyperplane with maximum margin. The hinge loss is:

$$
athcal{L} = rac{1}{N} um_{i=1}^{N} ax(0, 1 - y_i (w^T x_i + b)) + ambda w^2
$$

Here, labels are encoded as $y_i n -1, +1$.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

key = jax.random.PRNGKey(2)
key1, key2 = jax.random.split(key)
class0 = jax.random.normal(key1, (120, 2)) + jnp.array([-2.0, -1.5])
class1 = jax.random.normal(key2, (120, 2)) + jnp.array([2.0, 1.5])
x = jnp.concatenate([class0, class1], axis=0)
y = jnp.concatenate([jnp.ones((120, 1)) * -1.0, jnp.ones((120, 1))], axis=0)
perm = jax.random.permutation(key, x.shape[0])
x = x[perm]
y = y[perm]

In [ ]:
def scores(params, inputs):
    weight, bias = params
    return inputs @ weight + bias

def loss_fn(params, inputs, targets, reg_strength=0.01):
    margins = 1.0 - targets * scores(params, inputs)
    hinge = jnp.mean(jnp.maximum(0.0, margins))
    weight, _ = params
    regularization = reg_strength * jnp.sum(weight ** 2)
    return hinge + regularization

def accuracy(params, inputs, targets):
    preds = jnp.where(scores(params, inputs) >= 0.0, 1.0, -1.0)
    return jnp.mean(preds == targets)

def train(inputs, targets, learning_rate=0.05, steps=300):
    params = (jnp.zeros((2, 1)), jnp.zeros((1,)))
    grad_fn = jax.grad(loss_fn)
    history = []
    for step in range(steps):
        gradients = grad_fn(params, inputs, targets)
        params = tuple(param - learning_rate * grad for param, grad in zip(params, gradients))
        if step % 50 == 0 or step == steps - 1:
            history.append((step, float(loss_fn(params, inputs, targets)), float(accuracy(params, inputs, targets))))
    return params, history

params, history = train(x, y)
print(history)
print(f'final accuracy = {float(accuracy(params, x, y)):.3f}')

## Result

A trained SVM should produce a clear linear boundary between the two classes.

In [ ]:
weight, bias = params
grid_x, grid_y = jnp.meshgrid(jnp.linspace(-5.0, 5.0, 200), jnp.linspace(-5.0, 5.0, 200))
grid_points = jnp.stack([grid_x.ravel(), grid_y.ravel()], axis=1)
grid_scores = scores(params, grid_points).reshape(grid_x.shape)

plt.figure(figsize=(7, 6))
plt.contour(grid_x, grid_y, grid_scores, levels=[-1.0, 0.0, 1.0], colors=['gray', 'black', 'gray'], linestyles=['--', '-', '--'])
plt.scatter(x[:, 0], x[:, 1], c=y[:, 0], cmap='bwr', edgecolor='black', s=25)
plt.title('SVM in JAX')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()